# 03 — Verify Unity Catalog Setup
**Day 2 | Part 10**

Confirms the full Unity Catalog chain is working:
- Storage Credential (`ac-ev-intelligence-dev`) is registered
- External Locations (`bronze`, `silver`, `gold`) are created
- Schemas (`bronze`, `silver`, `gold`) exist in the catalog
- Volumes (`bronze_volume`, `silver_volume`, `gold_volume`) are browsable
- Files can be written/read via the `/Volumes/` path

**Prerequisites:**
- Part 10.1 — Access Connector created in Azure
- Part 10.2 — `Storage Blob Data Contributor` role assigned to Access Connector
- Part 10.3 — Storage Credential created in Databricks
- Part 10.4 — External Locations created (bronze, silver, gold)
- Part 10.5 — Schemas and Volumes created

## Cell 1 — Check External Locations are registered

In [ ]:
print("=== External Locations ===\n")

try:
    locations = spark.sql("SHOW EXTERNAL LOCATIONS").collect()

    if not locations:
        print("No external locations found.")
        print("Complete Part 10.4 — create External Locations in Catalog -> External Data -> External Locations")
    else:
        for loc in locations:
            name = loc["name"] if "name" in loc else loc[0]
            url  = loc["url"]  if "url"  in loc else loc[1]
            print(f"  {name:<25} -> {url}")

        required = {"bronze", "silver", "gold"}
        found    = {loc["name"] if "name" in loc else loc[0] for loc in locations}
        missing  = required - found

        if missing:
            print(f"\nMissing external locations: {missing}")
            print("Create them in: Databricks -> Catalog -> External Data -> External Locations")
        else:
            print(f"\nAll 3 required locations found (bronze, silver, gold) — OK")

except Exception as e:
    print(f"ERROR: {e}")
    print("Unity Catalog may not be enabled, or your user lacks SHOW EXTERNAL LOCATIONS privilege")

## Cell 2 — Check schemas in catalog

In [ ]:
CATALOG = "dbw_ev_intelligence_dev"

print(f"=== Schemas in {CATALOG} ===\n")

try:
    schemas = spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()
    schema_names = [s["databaseName"] if "databaseName" in s else s[0] for s in schemas]

    for name in schema_names:
        print(f"  {name}")

    required = {"bronze", "silver", "gold"}
    missing  = required - set(schema_names)

    if missing:
        print(f"\nMissing schemas: {missing}")
        print("Create them via SQL:")
        for s in missing:
            print(f"  CREATE SCHEMA IF NOT EXISTS {CATALOG}.{s};")
    else:
        print("\nAll 3 schemas found (bronze, silver, gold) — OK")

except Exception as e:
    print(f"ERROR: {e}")
    print(f"Catalog '{CATALOG}' may not exist or your user lacks access")

## Cell 3 — Browse volumes via dbutils.fs

In [ ]:
CATALOG = "dbw_ev_intelligence_dev"

for container in ["bronze", "silver", "gold"]:
    volume_path = f"/Volumes/{CATALOG}/{container}/{container}_volume/"
    print(f"\n=== {volume_path} ===")
    try:
        items = dbutils.fs.ls(volume_path)
        if items:
            for item in items:
                print(f"  {item.name}")
            print(f"  -> {len(items)} items found")
        else:
            print("  (empty — run 01_create_folder_structure notebook first)")
    except Exception as e:
        print(f"  ERROR: {e}")
        print(f"  -> Create the volume first: Part 10.5 in DAY2_STORAGE_SECURITY.md")

## Cell 4 — Write and read a test file through the Volume path

In [ ]:
CATALOG   = "dbw_ev_intelligence_dev"
test_path = f"/Volumes/{CATALOG}/bronze/bronze_volume/_uc_test.txt"

print("=== Volume write/read/delete test ===\n")
print(f"Test path: {test_path}\n")

try:
    dbutils.fs.put(test_path, "unity catalog volume write test ok", overwrite=True)
    print("Write  : OK")

    content = dbutils.fs.head(test_path)
    print(f"Read   : OK — '{content}'")

    dbutils.fs.rm(test_path)
    print("Delete : OK")

    print("\nVolume path read/write — PASSED")
    print("Files written here appear in the Catalog browser immediately.")

except Exception as e:
    print(f"ERROR: {e}")
    print("\nTroubleshooting:")
    print("  'Path does not exist'  -> Volume not created, or catalog/schema/volume name typo")
    print("  '403 Forbidden'        -> Access Connector missing Storage Blob Data Contributor role")
    print("  'Permission denied'    -> External Location test may have failed — re-check Part 10.4")

## Cell 5 — Full summary

In [ ]:
print("=" * 60)
print("UNITY CATALOG VERIFICATION — SUMMARY")
print("=" * 60)
print()
print("If all cells above showed OK or listed content:")
print("  [x] Storage Credential: ac-ev-intelligence-dev")
print("  [x] External Locations: bronze / silver / gold")
print("  [x] Schemas: dbw_ev_intelligence_dev.bronze / .silver / .gold")
print("  [x] Volumes: bronze_volume / silver_volume / gold_volume")
print("  [x] Volume write/read/delete via /Volumes/ path")
print()
print("Browse your storage in the Databricks UI:")
print("  Catalog (left menu) -> dbw_ev_intelligence_dev")
print("           -> bronze -> bronze_volume -> api/ -> payments/ ...")
print("           -> silver -> silver_volume")
print("           -> gold   -> gold_volume")
print()
print("Day 2 Unity Catalog setup complete — ready for Day 3.")
print("=" * 60)